# Validation against the FD reference 

Runs the selected CA-PINN configuration for three random seeds, runs the finite-difference reference solver, and computes the validation metrics, spatial errors and the mean and standard deviation over the seeds.

Configuration: tanh activation, 4 hidden layers with 40 neurons, 2000 Adam epochs followed by 10000 L-BFGS iterations, W_cons = 100, seeds 42, 1234 and 2024.

In [ ]:
# =========================
# 1. Imports and directory
# =========================

import os
import time
import gc
import random
import shutil

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

# Run this notebook from the repository root folder.

In [ ]:
# =====================================
# 2. Validation switches and parameters
# =====================================

# Execution switches
RUN_PREBIAS = False
RUN_FD      = False
RUN_COMPARE = False
MAKE_REPRESENTATIVE_FIGURE = True

# Final selected physical validation case
BASE_P0  = 1.0e21
BASE_PHI = 0.10

# Final selected PINN configuration
BASE_ACTIVATION    = "tanh"
BASE_HIDDEN_LAYERS = 4
BASE_NEURONS       = 40
BASE_WCONS         = 100.0

BASE_NUM_EPOCH_ADAM  = 2000

BASE_NUM_EPOCH_LBFGS = 10000
BASE_RUN_LBFGS       = True

# Independent random seeds
SEEDS = [42, 1234, 2024]

# Output root
BASE_DIR = "results/validation"
os.makedirs(BASE_DIR, exist_ok=True)

SUMMARY_PATH = os.path.join(
    BASE_DIR, "validation_all_seeds.csv"
)

AGG_PATH = os.path.join(
    BASE_DIR, "validation_mean_std.csv"
)

PUBLICATION_TABLE_PATH = os.path.join(
    BASE_DIR, "validation_publication_table.csv"
)

In [ ]:
# =====================
# 3. Helper functions
# =====================

LEFTOVERS = [
    "prebias_model", "main_model", "model", "layers", "results",
    "X_0", "X_r", "X_a", "X_c", "u_a", "u_c",
    "X_r_pre", "X_0_pre", "X_aPre", "X_cPre",
    "lb", "ub", "lb_pre", "ub_pre", "lb_main", "ub_main",
    "u0_interp_dict", "u_0", "x_0",
    "X_test", "X_list", "T_list",
    "psi_pred", "n_pred", "I_pred"
]

def set_all_seeds(seed):
    """Clear TensorFlow state and set all random seeds."""
    tf.keras.backend.clear_session()
    gc.collect()

    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

def require_files(outdir, filenames):
    """Raise an informative error when expected output files are missing."""
    missing = [
        name for name in filenames
        if not os.path.exists(os.path.join(outdir, name))
    ]
    if missing:
        raise FileNotFoundError(
            f"Missing required files in {outdir}: {missing}"
        )

def relative_l2(prediction, reference, eps=1e-300):
    prediction = np.asarray(prediction, dtype=float)
    reference  = np.asarray(reference, dtype=float)

    return (
        np.linalg.norm(prediction - reference)
        / (np.linalg.norm(reference) + eps)
    )

def rmse(prediction, reference):
    prediction = np.asarray(prediction, dtype=float)
    reference  = np.asarray(reference, dtype=float)

    return float(
        np.sqrt(np.mean((prediction - reference) ** 2))
    )

def load_validation_data(outdir):
    """Load PINN and FD outputs and interpolate them onto the FD potential grid."""
    psi_pinn = pd.read_csv(os.path.join(outdir, "pinn_psi.csv"))
    n_pinn   = pd.read_csv(os.path.join(outdir, "pinn_n.csv"))
    I_pinn   = pd.read_csv(os.path.join(outdir, "pinn_I.csv"))

    psi_fd = pd.read_csv(os.path.join(outdir, "fd_psi.csv"))
    n_fd   = pd.read_csv(os.path.join(outdir, "fd_n.csv"))
    I_fd   = pd.read_csv(os.path.join(outdir, "fd_I.csv"))

    x_nm = psi_fd["x_nm"].to_numpy(dtype=float)

    psi_p = np.interp(
        x_nm,
        psi_pinn["x_nm"].to_numpy(dtype=float),
        psi_pinn["psi_V"].to_numpy(dtype=float),
    )
    n_p = np.interp(
        x_nm,
        n_pinn["x_nm"].to_numpy(dtype=float),
        n_pinn["n_m3"].to_numpy(dtype=float),
    )
    I_p = np.interp(
        x_nm,
        I_pinn["x_nm"].to_numpy(dtype=float),
        I_pinn["I_m3"].to_numpy(dtype=float),
    )

    psi_f = psi_fd["psi_V"].to_numpy(dtype=float)
    n_f = np.interp(
        x_nm,
        n_fd["x_nm"].to_numpy(dtype=float),
        n_fd["n_m3"].to_numpy(dtype=float),
    )
    I_f = np.interp(
        x_nm,
        I_fd["x_nm"].to_numpy(dtype=float),
        I_fd["I_m3"].to_numpy(dtype=float),
    )

    return x_nm, psi_p, n_p, I_p, psi_f, n_f, I_f

def calculate_validation_metrics(outdir):
    """
    Calculate global metrics and save spatial errors.

    Pointwise errors:
      - psi: absolute error in volts
      - n and I: both absolute and stabilized relative errors
    """
    x_nm, psi_p, n_p, I_p, psi_f, n_f, I_f = load_validation_data(outdir)

    abs_psi = np.abs(psi_p - psi_f)
    abs_n   = np.abs(n_p - n_f)
    abs_I   = np.abs(I_p - I_f)

    # Stabilized pointwise relative errors.
    # The denominator floor prevents artificial divergence in very small regions.
    n_floor = max(1e-12 * np.max(np.abs(n_f)), 1e-300)
    I_floor = max(1e-12 * np.max(np.abs(I_f)), 1e-300)

    rel_n_point = abs_n / np.maximum(np.abs(n_f), n_floor)
    rel_I_point = abs_I / np.maximum(np.abs(I_f), I_floor)

    spatial_error = pd.DataFrame({
        "x_nm": x_nm,
        "psi_PINN_V": psi_p,
        "psi_FD_V": psi_f,
        "abs_error_psi_V": abs_psi,
        "n_PINN_m3": n_p,
        "n_FD_m3": n_f,
        "abs_error_n_m3": abs_n,
        "pointwise_rel_error_n": rel_n_point,
        "I_PINN_m3": I_p,
        "I_FD_m3": I_f,
        "abs_error_I_m3": abs_I,
        "pointwise_rel_error_I": rel_I_point,
    })

    spatial_path = os.path.join(outdir, "spatial_error_profiles.csv")
    spatial_error.to_csv(spatial_path, index=False)

    I_mean_pinn = float(np.trapezoid(I_p, x_nm) / (x_nm[-1] - x_nm[0]))
    I_mean_fd   = float(np.trapezoid(I_f, x_nm) / (x_nm[-1] - x_nm[0]))

    metrics = {
        "relL2_psi": relative_l2(psi_p, psi_f),
        "relL2_n": relative_l2(n_p, n_f),
        "relL2_I": relative_l2(I_p, I_f),

        "RMSE_psi_V": rmse(psi_p, psi_f),
        "RMSE_n_m3": rmse(n_p, n_f),
        "RMSE_I_m3": rmse(I_p, I_f),

        "MAE_psi_V": float(np.mean(abs_psi)),
        "MAE_n_m3": float(np.mean(abs_n)),
        "MAE_I_m3": float(np.mean(abs_I)),

        "max_abs_error_psi_V": float(np.max(abs_psi)),
        "max_abs_error_n_m3": float(np.max(abs_n)),
        "max_abs_error_I_m3": float(np.max(abs_I)),

        "Imean_PINN_m3": I_mean_pinn,
        "Imean_FD_m3": I_mean_fd,
        "Imean_rel_error": (
            abs(I_mean_pinn - I_mean_fd)
            / (abs(I_mean_fd) + 1e-300)
        ),
    }

    pd.DataFrame([metrics]).to_csv(
        os.path.join(outdir, "validation_metrics.csv"),
        index=False,
    )

    return metrics

def make_case_tag(seed):
    activation_tag = BASE_ACTIVATION.lower()

    return (
        f"P0_{BASE_P0:.0e}_phi{BASE_PHI:.2f}_"
        f"{activation_tag}_{BASE_HIDDEN_LAYERS}x{BASE_NEURONS}_"
        f"adam{BASE_NUM_EPOCH_ADAM}_"
        f"lbfgs{BASE_NUM_EPOCH_LBFGS}_seed{seed}"
    )

In [ ]:
# =======================================
# 4. Run three-seed PINN–FD validation
# =======================================

rows = []

for seed in SEEDS:

    tag = make_case_tag(seed)
    output_dir = os.path.join(BASE_DIR, tag)
    os.makedirs(output_dir, exist_ok=True)

    print("\n" + "=" * 92)
    print(f"Final PINN–FD validation case: {tag}")
    print(f"P0              = {BASE_P0:.3e} m^-3")
    print(f"phi             = {BASE_PHI:.2f} eV")
    print(f"activation      = {BASE_ACTIVATION}")
    print(f"architecture    = {BASE_HIDDEN_LAYERS} x {BASE_NEURONS}")
    print(f"W_cons          = {BASE_WCONS}")
    print(f"Adam epochs     = {BASE_NUM_EPOCH_ADAM}")
    print(f"L-BFGS iters    = {BASE_NUM_EPOCH_LBFGS}")
    print(f"RUN_LBFGS       = {BASE_RUN_LBFGS}")
    print(f"seed            = {seed}")
    print(f"OUTPUT_DIR      = {output_dir}")
    print("=" * 92)

    set_all_seeds(seed)

    notebook_globals = globals()

    for name in LEFTOVERS:
        notebook_globals.pop(name, None)

    # Pass parameters into the PINN and FD notebooks
    notebook_globals["P0"] = BASE_P0
    notebook_globals["phia"] = BASE_PHI
    notebook_globals["phic"] = BASE_PHI
    notebook_globals["OUTPUT_DIR"] = output_dir

    notebook_globals["ACTIVATION"] = BASE_ACTIVATION
    notebook_globals["HIDDEN_LAYERS"] = BASE_HIDDEN_LAYERS
    notebook_globals["NEURONS"] = BASE_NEURONS
    notebook_globals["W_cons"] = BASE_WCONS

    notebook_globals["NUM_EPOCH_ADAM"] = BASE_NUM_EPOCH_ADAM
    notebook_globals["NUM_EPOCH_LBFGS"] = BASE_NUM_EPOCH_LBFGS
    notebook_globals["RUN_LBFGS"] = BASE_RUN_LBFGS

    notebook_globals["SEED"] = seed

    case_start = time.time()

    try:
        if RUN_PREBIAS:
            stage_start = time.time()
            get_ipython().run_line_magic("run", '-i "CA_PINN_Solver.ipynb"')
            print(
                f"[{tag}] PINN runtime: "
                f"{time.time() - stage_start:.1f} s"
            )

        if RUN_FD:
            stage_start = time.time()
            get_ipython().run_line_magic("run", '-i "FD_Solver.ipynb"')
            print(
                f"[{tag}] FD runtime: "
                f"{time.time() - stage_start:.1f} s"
            )

        if RUN_COMPARE:
            require_files(
                output_dir,
                [
                    "pinn_psi.csv",
                    "pinn_n.csv",
                    "pinn_I.csv",
                    "fd_psi.csv",
                    "fd_n.csv",
                    "fd_I.csv",
                ],
            )

            metrics = calculate_validation_metrics(output_dir)

            row = {
                "P0_m3": float(BASE_P0),
                "phi_eV": float(BASE_PHI),
                "activation": BASE_ACTIVATION,
                "hidden_layers": int(BASE_HIDDEN_LAYERS),
                "neurons": int(BASE_NEURONS),
                "num_epoch_adam": int(BASE_NUM_EPOCH_ADAM),
                "num_epoch_lbfgs": int(BASE_NUM_EPOCH_LBFGS),
                "run_lbfgs": bool(BASE_RUN_LBFGS),
                "W_cons": float(BASE_WCONS),
                "seed": int(seed),
                "tag": tag,
                **metrics,
                "case_runtime_s": float(time.time() - case_start),
                "status": "ok",
            }

            rows.append(row)
            pd.DataFrame(rows).to_csv(SUMMARY_PATH, index=False)

            print(f"[{tag}] relL2(psi) = {metrics['relL2_psi']:.3e}")
            print(f"[{tag}] relL2(n)   = {metrics['relL2_n']:.3e}")
            print(f"[{tag}] relL2(I)   = {metrics['relL2_I']:.3e}")
            print(
                f"[{tag}] ion-mean relative error = "
                f"{metrics['Imean_rel_error']:.3e}"
            )

    except Exception as exc:
        print(f"[{tag}] FAILED: {repr(exc)}")

        rows.append({
            "P0_m3": float(BASE_P0),
            "phi_eV": float(BASE_PHI),
            "activation": BASE_ACTIVATION,
            "hidden_layers": int(BASE_HIDDEN_LAYERS),
            "neurons": int(BASE_NEURONS),
            "num_epoch_adam": int(BASE_NUM_EPOCH_ADAM),
            "num_epoch_lbfgs": int(BASE_NUM_EPOCH_LBFGS),
            "run_lbfgs": bool(BASE_RUN_LBFGS),
            "W_cons": float(BASE_WCONS),
            "seed": int(seed),
            "tag": tag,
            "case_runtime_s": float(time.time() - case_start),
            "status": f"failed: {type(exc).__name__}",
        })

        pd.DataFrame(rows).to_csv(SUMMARY_PATH, index=False)

# Save final per-seed table. When all switches are False (re-plot mode), no runs
# were computed and the saved table in results/validation/ is kept unchanged.
if rows:
    all_runs = pd.DataFrame(rows)
    all_runs.to_csv(SUMMARY_PATH, index=False)

print("\nCompleted all requested validation runs.")
print("Per-seed summary:", SUMMARY_PATH)

In [ ]:
# ====================================
# 5. Aggregate mean and standard deviation
# ====================================

all_runs = pd.read_csv(SUMMARY_PATH)
successful = all_runs[all_runs["status"] == "ok"].copy()

if len(successful) == 0:
    raise RuntimeError("No successful validation runs were found.")

metric_columns = [
    "relL2_psi",
    "relL2_n",
    "relL2_I",
    "RMSE_psi_V",
    "RMSE_n_m3",
    "RMSE_I_m3",
    "MAE_psi_V",
    "MAE_n_m3",
    "MAE_I_m3",
    "max_abs_error_psi_V",
    "max_abs_error_n_m3",
    "max_abs_error_I_m3",
    "Imean_PINN_m3",
    "Imean_FD_m3",
    "Imean_rel_error",
    "case_runtime_s",
]

aggregate_rows = []

for metric in metric_columns:
    aggregate_rows.append({
        "metric": metric,
        "mean": successful[metric].mean(),
        "std": successful[metric].std(ddof=1),
        "minimum": successful[metric].min(),
        "maximum": successful[metric].max(),
        "n_runs": len(successful),
    })

aggregate = pd.DataFrame(aggregate_rows)
aggregate.to_csv(AGG_PATH, index=False)

# Compact table (Table 7)
publication_rows = []

variable_map = {
    "Electrostatic potential, psi": "relL2_psi",
    "Electron density, n": "relL2_n",
    "Ion density, I": "relL2_I",
}

for variable, metric in variable_map.items():
    publication_rows.append({
        "variable": variable,
        "mean_relative_L2_error": successful[metric].mean(),
        "std_relative_L2_error": successful[metric].std(ddof=1),
        "n_runs": len(successful),
    })

publication_table = pd.DataFrame(publication_rows)
publication_table.to_csv(PUBLICATION_TABLE_PATH, index=False)

print("\nMean ± standard-deviation summary")
print(aggregate.to_string(index=False))

print("\nPublication validation table")
print(publication_table.to_string(index=False))

print("\nSaved:")
print(AGG_PATH)
print(PUBLICATION_TABLE_PATH)

In [ ]:
# =====================================================
# 6. Select a representative seed for profile plotting
# =====================================================

# Do not automatically plot the best seed.
# The representative run is selected as the seed whose aggregate relative-L2
# score is closest to the median score across successful runs.

successful = pd.read_csv(SUMMARY_PATH)
successful = successful[successful["status"] == "ok"].copy()

successful["aggregate_relL2"] = (
    successful["relL2_psi"]
    + successful["relL2_n"]
    + successful["relL2_I"]
) / 3.0

median_score = successful["aggregate_relL2"].median()

representative_index = (
    successful["aggregate_relL2"] - median_score
).abs().idxmin()

representative = successful.loc[representative_index]
representative_seed = int(representative["seed"])
representative_tag = representative["tag"]
representative_dir = os.path.join(BASE_DIR, representative_tag)

print("Representative seed:", representative_seed)
print("Representative tag:", representative_tag)
print(
    "Aggregate relative-L2 score:",
    f"{representative['aggregate_relL2']:.3e}",
)

pd.DataFrame([{
    "representative_seed": representative_seed,
    "representative_tag": representative_tag,
    "selection_rule": "aggregate relative-L2 score closest to three-seed median",
    "aggregate_relL2": representative["aggregate_relL2"],
}]).to_csv(
    os.path.join(BASE_DIR, "representative_seed_selection.csv"),
    index=False,
)

In [ ]:
# ==================================================
# 7. Publication figure with spatial-error profiles
# ==================================================

if MAKE_REPRESENTATIVE_FIGURE:

    spatial = pd.read_csv(
        os.path.join(
            representative_dir,
            "spatial_error_profiles.csv"
        )
    )

    # ==================================================
    # Load representative-run data
    # ==================================================

    x_nm = spatial["x_nm"].to_numpy(dtype=float)

    psi_p = spatial["psi_PINN_V"].to_numpy(dtype=float)
    psi_f = spatial["psi_FD_V"].to_numpy(dtype=float)

    n_p = spatial["n_PINN_m3"].to_numpy(dtype=float)
    n_f = spatial["n_FD_m3"].to_numpy(dtype=float)

    I_p = spatial["I_PINN_m3"].to_numpy(dtype=float)
    I_f = spatial["I_FD_m3"].to_numpy(dtype=float)

    # ==================================================
    # Spatial errors
    # ==================================================

    # Potential: dimensional absolute error
    err_psi = np.abs(
        psi_p - psi_f
    )

    # Electron and ion densities:
    # pointwise relative errors already stored in CSV
    err_n = spatial[
        "pointwise_rel_error_n"
    ].to_numpy(dtype=float)

    err_I = spatial[
        "pointwise_rel_error_I"
    ].to_numpy(dtype=float)

    # Replace non-positive values only for log plotting
    err_n_plot = np.where(
        err_n > 0.0,
        err_n,
        np.nan
    )

    err_I_plot = np.where(
        err_I > 0.0,
        err_I,
        np.nan
    )

    # ==================================================
    # Diagnostic checks
    # ==================================================

    positive_psi_error = (
        err_psi[
            err_psi > 0.0
        ]
    )

    print(
        "\nRepresentative-run validation diagnostics"
    )

    print(
        "-" * 55
    )

    print(
        f"Representative seed       : "
        f"{representative_seed}"
    )

    print(
        f"Maximum |psi_PINN-psi_FD| : "
        f"{np.max(err_psi):.6e} V"
    )

    print(
        f"Mean |psi_PINN-psi_FD|    : "
        f"{np.mean(err_psi):.6e} V"
    )

    if positive_psi_error.size > 0:

        print(
            "Minimum nonzero psi error : "
            f"{np.min(positive_psi_error):.6e} V"
        )

    else:

        print(
            "Minimum nonzero psi error : "
            "no nonzero values"
        )

    print(
        "Potential arrays identical : "
        f"{np.array_equal(psi_p, psi_f)}"
    )

    print(
        "Potential arrays all-close : "
        f"{np.allclose(psi_p, psi_f)}"
    )

    print(
        f"Maximum relative error n  : "
        f"{np.nanmax(err_n):.6e}"
    )

    print(
        f"Maximum relative error I  : "
        f"{np.nanmax(err_I):.6e}"
    )

    print(
        "-" * 55
    )

    # ==================================================
    # Plot settings
    # Match Figures 6 and 7
    # ==================================================

    plt.rcParams.update({

        "font.family":
            "serif",

        "mathtext.fontset":
            "dejavuserif",

        "font.size":
            13,

        "axes.labelsize":
            14,

        "axes.titlesize":
            14,

        "legend.fontsize":
            12,

        "xtick.labelsize":
            12,

        "ytick.labelsize":
            12,

        "axes.linewidth":
            1.0,

        "xtick.direction":
            "in",

        "ytick.direction":
            "in",

        "xtick.top":
            True,

        "ytick.right":
            True,

        "xtick.major.size":
            4,

        "ytick.major.size":
            4,

        "xtick.minor.size":
            2,

        "ytick.minor.size":
            2,

        "lines.linewidth":
            2.2,

        "savefig.dpi":
            600,

        "savefig.bbox":
            "tight",
    })

    # ==================================================
    # Create figure
    # ==================================================

    fig, axes = plt.subplots(
        nrows=2,
        ncols=3,
        figsize=(13.8, 7.8),
        sharex=True,
    )

    # ==================================================
    # Top row: FD and PINN solution profiles
    # ==================================================

    # --------------------------------------------------
    # (a) Electrostatic potential
    # --------------------------------------------------

    axes[0, 0].plot(
        x_nm,
        psi_f,
        linewidth=2.2,
        label="FD",
    )

    axes[0, 0].plot(
        x_nm,
        psi_p,
        "--",
        linewidth=2.0,
        label="PINN",
    )

    axes[0, 0].set_ylabel(
        r"$\psi$ (V)"
    )

    # --------------------------------------------------
    # (b) Electron density
    # --------------------------------------------------

    axes[0, 1].semilogy(
        x_nm,
        n_f,
        linewidth=2.2,
        label="FD",
    )

    axes[0, 1].semilogy(
        x_nm,
        n_p,
        "--",
        linewidth=2.0,
        label="PINN",
    )

    axes[0, 1].set_ylabel(
        r"$n$ (m$^{-3}$)"
    )

    # --------------------------------------------------
    # (c) Ion density
    # --------------------------------------------------

    axes[0, 2].semilogy(
        x_nm,
        I_f,
        linewidth=2.2,
        label="FD",
    )

    axes[0, 2].semilogy(
        x_nm,
        I_p,
        "--",
        linewidth=2.0,
        label="PINN",
    )

    axes[0, 2].set_ylabel(
        r"$I$ (m$^{-3}$)"
    )

    # ==================================================
    # Bottom row: spatial errors
    # ==================================================

    # --------------------------------------------------
    # (d) Potential absolute error
    # --------------------------------------------------

    if (
        positive_psi_error.size > 0
        and
        np.max(
            positive_psi_error
        ) >= 1.0e-12
    ):

        err_psi_plot = np.where(
            err_psi > 0.0,
            err_psi,
            np.nan,
        )

        axes[1, 0].semilogy(
            x_nm,
            err_psi_plot,
            linewidth=2.2,
        )

    else:

        axes[1, 0].plot(
            x_nm,
            err_psi,
            linewidth=2.2,
        )

        axes[1, 0].ticklabel_format(
            axis="y",
            style="sci",
            scilimits=(0, 0),
        )

    axes[1, 0].set_ylabel(
        r"Absolute error in $\psi$ (V)"
    )

    # --------------------------------------------------
    # (e) Electron-density relative error
    # --------------------------------------------------

    axes[1, 1].semilogy(
        x_nm,
        err_n_plot,
        linewidth=2.2,
    )

    axes[1, 1].set_ylabel(
        r"Relative error in $n$"
    )

    # --------------------------------------------------
    # (f) Ion-density relative error
    # --------------------------------------------------

    axes[1, 2].semilogy(
        x_nm,
        err_I_plot,
        linewidth=2.2,
    )

    axes[1, 2].set_ylabel(
        r"Relative error in $I$"
    )

    # ==================================================
    # Panel labels
    # Match Figures 6 and 7
    # ==================================================

    panel_labels = [
        "(a)",
        "(b)",
        "(c)",
        "(d)",
        "(e)",
        "(f)",
    ]

    for axis, label in zip(
        axes.flat,
        panel_labels
    ):

        axis.text(
            0.02,
            1.03,
            label,
            transform=axis.transAxes,
            ha="left",
            va="bottom",
            fontsize=11,
            fontweight="bold",
            clip_on=False,
        )

    # ==================================================
    # Axes formatting
    # ==================================================

    for row in range(2):

        for col in range(3):

            axis = axes[
                row,
                col
            ]

            axis.set_xlim(
                x_nm.min(),
                x_nm.max(),
            )

            axis.grid(
                True,
                which="major",
                linestyle=":",
                linewidth=0.6,
                alpha=0.5,
            )

            axis.grid(
                True,
                which="minor",
                linestyle=":",
                linewidth=0.4,
                alpha=0.3,
            )

    # Bottom-row x labels
    for col in range(3):

        axes[1, col].set_xlabel(
            "Position (nm)"
        )

    # ==================================================
    # Shared legend
    # ==================================================

    handles, labels = (
        axes[0, 0]
        .get_legend_handles_labels()
    )

    fig.legend(
        handles,
        labels,
        loc="upper center",
        ncol=2,
        frameon=False,
        bbox_to_anchor=(
            0.5,
            0.995
        ),
        handlelength=2.8,
        columnspacing=1.6,
        handletextpad=0.6,
    )

    # ==================================================
    # Layout
    # ==================================================

    fig.subplots_adjust(
        left=0.075,
        right=0.985,
        bottom=0.10,
        top=0.88,
        wspace=0.30,
        hspace=0.28,
    )

    # ==================================================
    # Save publication figure
    # ==================================================

    figure_pdf = os.path.join(
        BASE_DIR,
        "validation_with_spatial_errors.pdf",
    )

    figure_png = os.path.join(
        BASE_DIR,
        "validation_with_spatial_errors.png",
    )

    fig.savefig(
        figure_pdf,
        format="pdf",
        bbox_inches="tight",
    )

    fig.savefig(
        figure_png,
        format="png",
        dpi=600,
        bbox_inches="tight",
    )

    plt.show()

    plt.close(
        fig
    )

    print(
        "\nSaved publication figures:"
    )

    print(
        figure_pdf
    )

    print(
        figure_png
    )

## Outputs

1. One folder per seed with the PINN and FD profiles, `validation_metrics.csv` and `spatial_error_profiles.csv`.
2. `validation_all_seeds.csv`, `validation_mean_std.csv` and `validation_publication_table.csv` (Table 7).
3. `validation_with_spatial_errors.pdf` (Fig. 3), plotted for the representative seed.
4. `representative_seed_selection.csv`: the representative seed is the one whose aggregate relative L2 error is closest to the three-seed median.